# Toolbox Tool Search `@azure/ai-projects`

This notebook demonstrates how to add the generally available toolbox search tool (`toolbox_search`) to a toolbox, and how to let an agent discover and call the toolbox's tools through it.

A toolbox exposes its tools over an MCP endpoint. Adding the `toolbox_search` tool lets an agent search that toolbox for a relevant tool instead of having every tool declared up front. In the JS SDK, you access these operations via `project.toolboxes`.

It mirrors the [`toolboxToolSearch.ts`](./toolboxToolSearch.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. **Build the package first** so `dist/` is current, from the repo root: `pnpm turbo build --filter=@azure/ai-projects... --token 1`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_ENDPOINT`, `FOUNDRY_MODEL_NAME`.

Run the cells in order (top to bottom); state is shared across cells.

In [1]:
// Imports and configuration
import type { MCPTool, MCPToolboxTool, ToolboxToolUnion } from "@azure/ai-projects";
import { AIProjectClient } from "@azure/ai-projects";
import { DefaultAzureCredential } from "@azure/identity";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";
const deploymentName = process.env["FOUNDRY_MODEL_NAME"] ?? "<model deployment name>";
const toolboxName = "toolbox-search-sample";

In [2]:
// Create the AI Project client
const credential = new DefaultAzureCredential();
const project = new AIProjectClient(projectEndpoint, credential);

In [3]:
// A toolbox holds the tools an agent can reach. Adding `toolbox_search` alongside them
// lets the agent search the toolbox at run time rather than binding every tool up front.
const tools: ToolboxToolUnion[] = [
  {
    type: "mcp",
    server_label: "api_specs",
    server_url: "https://gitmcp.io/Azure/azure-rest-api-specs",
    require_approval: "never",
  } satisfies MCPToolboxTool,
  {
    type: "toolbox_search",
  },
];

const version: any = await project.toolboxes.createVersion(toolboxName, tools, {
  description: "Example toolbox with tool search enabled.",
});
console.log(`Toolbox version created: ${version.name}:${version.version}`);

Toolbox version created: toolbox-search-sample:1


In [4]:
// Point the agent at the toolbox's MCP endpoint. The agent calls `tool_search` to find a
// tool, then `call_tool` to invoke the one it picked.
const toolboxMcpUrl = `${projectEndpoint}/toolboxes/${version.name}/versions/${version.version}/mcp?api-version=v1`;
const token = (await credential.getToken("https://ai.azure.com/.default")).token;

const toolboxMcpTool: MCPTool = {
  type: "mcp",
  server_label: "toolbox_search",
  server_url: toolboxMcpUrl,
  authorization: token,
  require_approval: "never",
};

In [5]:
// Create a prompt agent pointed at the toolbox MCP tool
const agent: any = await project.agents.createVersion("toolbox-search-sample-agent", {
  kind: "prompt",
  model: deploymentName,
  instructions:
    "Always use the toolbox search tool to answer questions. " +
    "Call `tool_search` to discover a relevant tool, then `call_tool` " +
    "with the tool name returned by the search.",
  tools: [toolboxMcpTool],
});
console.log(`Agent created: ${agent.name}:${agent.version}`);

Agent created: toolbox-search-sample-agent:1


In [6]:
// Ask the agent a question it can only answer by searching the toolbox for a tool.
const openAIClient: any = project.getOpenAIClient();
console.log("\nGenerating response...");
const response: any = await openAIClient.responses.create(
  {
    input: "Which Azure REST API specs describe the Foundry data plane?",
  },
  {
    body: { agent_reference: { name: agent.name, type: "agent_reference" } },
  },
);
console.log(`Response output: ${response.output_text}`);


Generating response...
Response output: The **Foundry data-plane** REST API specs in `Azure/azure-rest-api-specs` are described by the **AI Projects** data-plane OpenAPI specs (generated from the Foundry TypeSpec sources):

- TypeSpec source root (Foundry data-plane):  
  `specification/ai-foundry/data-plane/Foundry/`

- Published/generated OpenAPI (AI Projects data-plane), referenced by the AutoRest readme:  
  `specification/ai/data-plane/Azure.AI.Projects/stable/v1/azure-ai-projects.json`  
  `specification/ai/data-plane/Azure.AI.Projects/stable/2025-05-01/azure-ai-projects.json`  
  `specification/ai/data-plane/Azure.AI.Projects/stable/2025-05-15-preview/azure-ai-projects.json`

- AutoRest configuration that enumerates those specs:  
  `specification/ai/data-plane/Azure.AI.Projects/readme.md`


In [7]:
// Clean up
console.log("\nCleaning up resources...");
await project.agents.deleteVersion(agent.name, agent.version);
await project.toolboxes.deleteVersion(version.name, version.version);
console.log("Agent and toolbox version deleted");


Cleaning up resources...
Agent and toolbox version deleted
